# A — Phân tích dữ liệu HumBugDB 0.0.2

**Mục tiêu:** hiểu dữ liệu trước khi thảo luận chọn 4–6 lớp cho bài toán microphone.
Notebook chạy local, khảo sát toàn bộ dataset và lưu kết quả vào **results/A**.

### Cách đọc notebook
Mỗi phần có mục đích, phương pháp, biểu đồ và nhận xét tính từ dữ liệu.
Chạy từ trên xuống bằng **Run All**. Lần đầu sẽ đọc toàn bộ WAV để kiểm tra chất lượng;
lần sau dùng cache nếu file và cấu hình kiểm tra không đổi.

1. Nguồn dữ liệu và kiểm tra ánh xạ.
2. Tổng quan, metadata thiếu và phân vùng gốc.
3. Thống kê từng nhãn, độ dài và khả năng tạo window.
4. Nguồn ghi, mức tập trung và điều kiện thu.
5. Kiểm tra toàn bộ audio và dữ liệu trùng.
6. Nghe mẫu, waveform và spectrogram.
7. Bảng ứng viên để thảo luận.
8. Tổng hợp, giới hạn và report.

**Phạm vi:** đây là phần khảo sát trước khi chọn lớp của A. Chưa tạo split nghiên cứu,
chưa loại file và chưa huấn luyện AI. A4 (split), quyết định đầu vào A5 và giao thức
cuối A6 sẽ hoàn thiện sau khi thống nhất lớp và căn cứ nhóm nguồn.

**Đơn vị:** một clip WAV không đồng nghĩa một cá thể. Tổng độ dài clip có thể đếm lại
thời gian nếu các clip chồng lấn; chưa có timestamp đoạn cắt để tính thời lượng gốc duy nhất.
Tên nguồn trong cột name chỉ là dấu hiệu truy vết, chưa chứng minh độc lập cá thể/session.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone, timedelta
import base64, hashlib, html, io, json, math, platform, time
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
from scipy import signal
from IPython.display import Audio, Markdown, display
import mistune

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config.json").is_file() and (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục project hoặc notebooks.")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
OUT = ROOT / CONFIG["results_dir"] / "A"
for folder in [OUT / "tables", OUT / "figures"]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
WINDOW_SECONDS = 0.300
HOP_SECONDS = 0.150
NEAR_SILENCE_DBFS = -60.0       # Cờ khảo sát, không phải ngưỡng loại.
NEAR_FULL_SCALE = 0.999        # Gần full scale, chưa chứng minh méo/clipping.
FORCE_AUDIO_SCAN = False
PREVIEW_SPECIES = []           # Điền nhãn muốn nghe; để trống = 6 nhãn ngẫu nhiên.
QC_VERSION = 1
SECTION = 0
REPORT_ITEMS = []
RUN_TIME = datetime.now(timezone(timedelta(hours=7))).isoformat(timespec="seconds")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
sns.set_theme(style="whitegrid", context="notebook", font="DejaVu Sans")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlesize": 12})

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def note(text):
    display(Markdown(text))
    REPORT_ITEMS.append((SECTION, "text", text, ""))

def table(frame, name, caption, rows=45):
    frame.to_csv(OUT / "tables" / f"{name}.csv", index=False, encoding="utf-8-sig")
    display(Markdown(f"**{caption}** — {len(frame):,} dòng; CSV lưu đầy đủ."))
    display(frame.head(rows))
    REPORT_ITEMS.append((SECTION, "table", frame.head(rows).copy(), (name, caption, len(frame))))

def figure(fig, name, caption):
    fig.tight_layout()
    path = OUT / "figures" / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    display(Markdown(caption))
    REPORT_ITEMS.append((SECTION, "figure", path, caption))

def category(series):
    return series.replace("", "(thiếu)")

versions = {p: package_metadata.version(p) for p in
            ["numpy", "pandas", "scipy", "soundfile", "matplotlib", "seaborn", "nbformat", "nbconvert", "ipykernel"]}
note(f"Thời điểm chạy: **{RUN_TIME}**. Python **{platform.python_version()}**. "
     "Các ngưỡng ở trên chỉ gắn cờ kiểm tra; không tự xóa hoặc lọc WAV.")

## 1. Nguồn dữ liệu và kiểm tra ánh xạ

**Làm gì:** đọc metadata gốc và inventory; join theo ID với kiểm tra một–một.
Đối chiếu file thực tế, kích thước và đường dẫn; lưu dấu vết SHA-256 của đầu vào.

**Đọc kết quả:** thiếu/dư ID, đường dẫn ngoài thư mục raw hoặc sai kích thước phải được
giải quyết trước khi dùng bảng thống kê. SHA-256 audio và header sẽ được kiểm tra ở phần 5.
Nguồn phát hành: [HumBugDB 0.0.2](https://zenodo.org/records/6478589).
Giữ nguyên train/dev/a/dev/b của gói gốc; chưa diễn giải a/b thành validation/test.

In [ ]:
SECTION = 1
metadata_path = ROOT / CONFIG["metadata_csv"]
inventory_path = ROOT / CONFIG["inventory"]
raw_root = (ROOT / CONFIG["audio_dir"]).resolve()
meta = pd.read_csv(metadata_path, dtype=str, keep_default_na=False)
meta = meta.apply(lambda s: s.str.strip())
inv = pd.read_json(inventory_path, lines=True, dtype={"id": str})
inv["id"] = inv["id"].astype(str)
assert meta["id"].is_unique and inv["id"].is_unique, "ID bị trùng."
missing_ids = sorted(set(meta.id) - set(inv.id))
extra_ids = sorted(set(inv.id) - set(meta.id))
assert not missing_ids and not extra_ids, (missing_ids[:10], extra_ids[:10])
meta = meta.rename(columns={"sample_rate": "metadata_sample_rate", "length": "metadata_length"})
meta["metadata_sample_rate"] = pd.to_numeric(meta["metadata_sample_rate"], errors="coerce")
meta["metadata_length"] = pd.to_numeric(meta["metadata_length"], errors="coerce")
df = meta.merge(inv, on="id", how="inner", validate="one_to_one")
df["partition"] = df["relative_path"].str.replace("data/raw/", "", regex=False).str.rsplit("/", n=1).str[0]
df["label"] = category(df["species"])
df["source"] = df["name"].replace("", pd.NA)

file_states = []
for row in df.itertuples():
    path = (ROOT / row.relative_path).resolve()
    assert path.is_relative_to(raw_root), path
    stat = path.stat()
    file_states.append((row.id, stat.st_size, stat.st_mtime_ns))
states = pd.DataFrame(file_states, columns=["id", "actual_bytes", "mtime_ns"])
df = df.merge(states, on="id", validate="one_to_one")
actual_paths = {p.relative_to(ROOT).as_posix() for p in raw_root.rglob("*") if p.is_file()}
assert actual_paths == set(df.relative_path), "Có file thiếu/dư trong raw."
assert (df["bytes"] == df["actual_bytes"]).all(), "Kích thước WAV đã thay đổi."
df["metadata_rate_matches"] = df.metadata_sample_rate.eq(df.sample_rate)
df["metadata_duration_matches_now"] = (
    (df.metadata_length - df.duration_seconds).abs() <= np.maximum(.001, 1/df.sample_rate))
metadata_mismatch = df[~df.metadata_rate_matches | ~df.metadata_duration_matches_now]
table(metadata_mismatch[["id", "metadata_sample_rate", "sample_rate", "metadata_length",
                        "duration_seconds"]], "metadata_audio_mismatch", "Sai khác metadata và header trong inventory")
assert metadata_mismatch.empty, "Metadata không khớp header; cần kiểm tra trước khi phân tích."
input_hashes = {"metadata": sha256(metadata_path), "inventory": sha256(inventory_path)}
table(pd.DataFrame([
    {"kiểm tra": "Dòng metadata", "kết quả": len(meta)},
    {"kiểm tra": "WAV thực có", "kết quả": len(actual_paths)},
    {"kiểm tra": "ID thiếu", "kết quả": len(missing_ids)},
    {"kiểm tra": "ID dư", "kết quả": len(extra_ids)},
    {"kiểm tra": "Sai kích thước so với inventory", "kết quả": int((df.bytes != df.actual_bytes).sum())}
]), "input_checks", "Đối chiếu đầu vào")
table(df[["id", "species", "sound_type", "name", "relative_path", "duration_seconds",
          "sample_rate", "channels"]].head(12), "input_examples", "Ví dụ ánh xạ ID sang WAV")
note(f"Có **{len(df):,} WAV**, tổng **{df.bytes.sum()/1e9:.2f} GB**. "
     "Metadata chỉ được làm sạch khoảng trắng trong bộ nhớ; CSV và audio gốc giữ nguyên.")

## 2. Tổng quan và độ đầy đủ của metadata

**Làm gì:** tách các loại sound_type; so số file với tổng độ dài clip. Kiểm tra phân vùng,
sample rate, channels và tỷ lệ thiếu từng trường. Chỉ các dòng mosquito có species mới
được đưa vào bảng ứng viên phân loại loài.

**Đọc biểu đồ:** nhiều file chưa chắc nhiều thời lượng. Trục độ dài dùng log vì có cả
clip rất ngắn và bản ghi dài. Các nhãn audio/false_positive/background được giữ riêng;
chưa tự coi chúng là ba loại negative tương đương.

In [ ]:
SECTION = 2
sound_summary = df.groupby("sound_type", dropna=False).agg(
    clips=("id", "size"), seconds=("duration_seconds", "sum"),
    median_seconds=("duration_seconds", "median")).reset_index()
sound_summary["hours"] = sound_summary.seconds / 3600
table(sound_summary, "sound_types", "Số clip và thời lượng theo loại âm thanh")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(sound_summary.sound_type, sound_summary.clips, color="#326b98")
axes[0].set(title="Số clip theo sound_type", ylabel="Clip")
axes[1].bar(sound_summary.sound_type, sound_summary.hours, color="#338777")
axes[1].set(title="Tổng độ dài clip", ylabel="Giờ")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
figure(fig, "01_sound_types", "Hai biểu đồ dùng cùng nhóm nhưng khác đơn vị. Giờ ở đây là tổng độ dài clip, chưa khử phần chồng lấn.")

partition_summary = df.groupby(["partition", "sound_type"]).agg(
    clips=("id", "size"), seconds=("duration_seconds", "sum")).reset_index()
table(partition_summary, "partitions", "Nội dung các phân vùng gốc")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.crosstab(df.partition, df.sound_type).plot.bar(stacked=True, ax=axes[0], rot=0)
axes[0].set(title="Phân vùng và loại âm thanh", xlabel="", ylabel="Clip")
axes[0].legend(title="sound_type", fontsize=8)
df.sample_rate.value_counts().sort_index().plot.bar(ax=axes[1], color="#326b98", rot=0)
axes[1].set(title="Sample rate gốc", xlabel="Hz", ylabel="Clip")
figure(fig, "02_partitions_rates", "Phân vùng là thông tin nguồn, không phải split nghiên cứu đã được chọn.")

fields = ["species", "name", "gender", "fed", "plurality", "age", "method", "mic_type",
          "device_type", "country", "district", "province", "place", "location_type"]
mosquito_all = df[df.sound_type.eq("mosquito")].copy()
missing = pd.DataFrame({"field": fields,
    "missing_all_pct": [(df[f].eq("").mean()*100) for f in fields],
    "missing_mosquito_pct": [(mosquito_all[f].eq("").mean()*100) for f in fields]})
table(missing, "metadata_missing", "Tỷ lệ thiếu metadata: toàn bộ và riêng mosquito")
fig, ax = plt.subplots(figsize=(10, 6))
missing.set_index("field").sort_values("missing_mosquito_pct").plot.barh(ax=ax, color=["#b8c6d0", "#326b98"])
ax.set(xlabel="Thiếu (%)", ylabel="", title="Không đồng nhất độ đầy đủ metadata")
ax.legend(["Toàn bộ", "Mosquito"], loc="lower right")
figure(fig, "03_missing_metadata", "Species trống trong background có thể hợp lý. Species trống trong mosquito là vấn đề riêng cần xem xét.")

fig, ax = plt.subplots(figsize=(10, 4))
for name, group in df.groupby("sound_type"):
    values = group.loc[group.duration_seconds.gt(0), "duration_seconds"]
    ax.hist(values, bins=np.geomspace(max(df.duration_seconds[df.duration_seconds.gt(0)].min(), 1e-5),
                                    df.duration_seconds.max()*1.01, 55),
            histtype="step", linewidth=1.6, label=name)
ax.set(xscale="log", yscale="log", xlabel="Độ dài clip (giây, log)", ylabel="Số clip (log)",
       title="Phân bố độ dài theo loại âm thanh")
ax.axvline(WINDOW_SECONDS, color="#b54b3f", linestyle="--", label="300 ms")
ax.legend()
figure(fig, "04_duration_overview", "Đường 300 ms giúp nhìn phần dữ liệu ngắn hơn window dự kiến; không phải ngưỡng tự loại.")
note(f"Trong **{len(mosquito_all):,}** clip mosquito, "
     f"**{mosquito_all.species.eq('').sum():,}** clip thiếu species. "
     f"Có **{df.channels.nunique()}** giá trị số kênh: **{sorted(df.channels.unique().tolist())}**.")

## 3. Từng nhãn loài: lượng dữ liệu, độ dài và window

**Làm gì:** giữ đúng chuỗi species; đếm số clip, tổng giây, median và các phân vị.
Tính số window đầy đủ cho các độ dài khác nhau; không tạo file window.

**Cách tính:** với N sample, window L sample và bước H sample, số window đầy đủ là
max(0, 1 + floor((N − L)/H)). Không padding, không ghép hai clip. Các window overlap
có tương quan; số window không thay thế số bản ghi hoặc cá thể độc lập.

Nhãn có complex, sl hoặc sp cần rà soát phân loại học trước khi gọi tất cả là “loài”.
Không tự đổi tên hoặc gộp nhãn trong notebook này.

In [ ]:
SECTION = 3
mosq = df[df.sound_type.eq("mosquito") & df.species.ne("")].copy()
species = mosq.groupby("species").agg(
    clips=("id", "size"), total_seconds=("duration_seconds", "sum"),
    unique_audio_hashes=("sha256", "nunique"),
    median_seconds=("duration_seconds", "median"), min_seconds=("duration_seconds", "min"),
    max_seconds=("duration_seconds", "max"), source_names=("source", "nunique"),
    missing_source_clips=("name", lambda s: int(s.eq("").sum()))
).sort_values("clips", ascending=False)
species["p10_seconds"] = mosq.groupby("species").duration_seconds.quantile(.1)
species["p90_seconds"] = mosq.groupby("species").duration_seconds.quantile(.9)
species["short_300ms_pct"] = mosq.groupby("species").duration_seconds.apply(lambda s: s.lt(.3).mean()*100)
species["label_review"] = species.index.to_series().str.contains(r"complex|\bsl\b|\bsp\b", regex=True)
species["minutes"] = species.total_seconds / 60
table(species.reset_index(), "species_summary", "Tất cả nhãn mosquito có species", rows=60)
order = species.index[::-1]
fig, axes = plt.subplots(1, 3, figsize=(16, max(8, len(order)*.29)), sharey=True)
for ax, col, title, color in zip(axes, ["clips", "minutes", "source_names"],
                                ["Số clip", "Tổng phút clip", "Số tên nguồn khác nhau"],
                                ["#326b98", "#338777", "#a47733"]):
    bars = ax.barh(order, species.loc[order, col], color=color)
    ax.bar_label(bars, labels=[f"{v:.1f}" if col == "minutes" else f"{v:.0f}"
                              for v in species.loc[order, col]], fontsize=7, padding=3)
    ax.margins(x=.14)
    ax.set(title=title, xlabel=title)
    ax.tick_params(axis="y", labelsize=9)
figure(fig, "05_species_volume", "Đọc cả ba cột cùng nhau. Source_names là số name khác nhau, chưa phải số cá thể/session.")
fig, ax = plt.subplots(figsize=(11, max(8, len(species)*.29)))
box = ax.boxplot([mosq.loc[mosq.species.eq(label), "duration_seconds"].to_numpy() for label in order],
                 orientation="horizontal", tick_labels=order, whis=(10, 90),
                 showfliers=False, patch_artist=True)
for patch in box["boxes"]:
    patch.set_facecolor("#c3d9e8")
ax.axvline(.3, color="#b54b3f", linestyle="--", label="300 ms")
ax.set(xscale="log", xlabel="Độ dài clip (giây, log)", title="Độ dài clip trong từng nhãn")
ax.legend()
figure(fig, "14_species_duration", "Hộp: phân vị 25–75%, đường giữa: median; râu hướng đến phân vị 10–90%. Ngoại lệ không vẽ, vẫn có trong bảng CSV.")
table(pd.crosstab(mosq.species, mosq.sample_rate).reset_index(), "species_sample_rates",
      "Sample rate theo nhãn — kiểm tra khác biệt đầu vào", rows=60)

window_rows = []
for seconds in [.1, .3, .625, 1., 2.]:
    length = np.maximum(1, np.rint(mosq.sample_rate.to_numpy()*seconds).astype(np.int64))
    hop = np.maximum(1, np.rint(mosq.sample_rate.to_numpy()*seconds/2).astype(np.int64))
    counts = np.maximum(0, 1 + (mosq.frames.to_numpy()-length)//hop)
    temp = mosq[["species", "id"]].copy()
    temp["windows"] = counts
    temp["eligible"] = counts > 0
    agg = temp.groupby("species").agg(clips=("id", "size"), eligible_clips=("eligible", "sum"),
                                     windows_50pct_overlap=("windows", "sum")).reset_index()
    agg["window_seconds"] = seconds
    agg["eligible_pct"] = agg.eligible_clips / agg.clips * 100
    window_rows.append(agg)
window_table = pd.concat(window_rows, ignore_index=True)
table(window_table, "window_feasibility", "Khả năng tạo window đầy đủ theo nhãn và độ dài", rows=12)
fig, ax = plt.subplots(figsize=(8, max(8, len(species)*.29)))
sns.heatmap(window_table.pivot(index="species", columns="window_seconds", values="eligible_pct")
            .reindex(species.index), vmin=0, vmax=100, cmap="YlGnBu", annot=True, fmt=".0f",
            linewidths=.3, ax=ax, cbar_kws={"label": "% clip có ít nhất 1 window"})
ax.set(title="Độ phủ clip theo độ dài window", xlabel="Window (giây), overlap 50%", ylabel="")
figure(fig, "06_window_coverage", "Ô 100% chỉ nói clip đủ dài. Chưa xác nhận window chứa tiếng muỗi rõ hoặc có nhãn đúng ở mọi thời điểm.")
table(pd.crosstab(mosq.species, mosq.partition).reset_index(), "species_partitions",
      "Số clip mỗi nhãn trong train/dev/a/dev/b", rows=60)
note(f"Có **{len(species)} chuỗi nhãn** ở **{len(mosq):,} clip mosquito có species**. "
     f"**{int(mosq.duration_seconds.lt(.3).sum())}** clip trong nhóm này ngắn hơn 300 ms. "
     "Đây là nhóm có nhãn; số clip ngắn của toàn bộ mosquito có thể khác.")

## 4. Nguồn ghi và nguy cơ học dấu hiệu của điều kiện thu

### 4.1. Nhiều clip có đến từ rất ít nguồn?
Tính tỷ lệ thời lượng clip thuộc name lớn nhất và số nguồn hiệu dụng theo thời lượng
(1 / tổng bình phương tỷ trọng nguồn). Nếu một name chiếm gần hết thời lượng,
số clip lớn vẫn chưa tạo được đánh giá độc lập mạnh. Nguồn hiệu dụng tính trên các
name đã biết; đây là thống kê độ tập trung, không phải số cá thể.

### 4.2. Loài có gắn với thiết bị/phương pháp thu?
Heatmap theo hàng cho thấy tỷ lệ clip của mỗi nhãn trong từng điều kiện. Một hàng
tập trung vào một thiết bị là dấu hiệu cần kiểm tra confounding. Đây là mô tả dữ liệu,
không phải bằng chứng model đã học nhầm. Các cột ít mẫu được gộp “khác” chỉ để vẽ;
bảng CSV vẫn giữ danh mục gốc.

### 4.3. Metadata sinh học và nguồn
Đọc riêng gender, plurality, fed, age, phương pháp, thiết bị và địa điểm. Thiếu plurality
không đồng nghĩa Single. upload_time là thời điểm upload, không dùng thay ngày thu.
Name xuất hiện ở nhiều nhãn/điều kiện cần truy vết, chưa tự kết luận nhãn sai.

In [ ]:
SECTION = 4
known = mosq[mosq.name.ne("")].copy()
source_table = known.groupby(["species", "name"]).agg(
    clips=("id", "size"), seconds=("duration_seconds", "sum")).reset_index()
concentration_rows = []
for label, group in mosq.groupby("species"):
    sources = source_table[source_table.species.eq(label)]
    known_seconds = sources.seconds.sum()
    weights = sources.seconds / known_seconds if known_seconds > 0 else pd.Series(dtype=float)
    concentration_rows.append({
        "species": label, "source_names": len(sources),
        "known_source_duration_pct": 100*known_seconds/group.duration_seconds.sum(),
        "largest_source_duration_pct": 100*sources.seconds.max()/group.duration_seconds.sum() if len(sources) else np.nan,
        "effective_sources_known": 1/float((weights**2).sum()) if len(weights) else np.nan,
        "largest_source_name": sources.sort_values("seconds", ascending=False).iloc[0]["name"] if len(sources) else ""
    })
concentration = pd.DataFrame(concentration_rows).set_index("species")
table(source_table.sort_values("seconds", ascending=False), "source_counts",
      "Clip và thời lượng theo cặp nhãn–name", rows=20)
table(concentration.reset_index().sort_values("largest_source_duration_pct", ascending=False),
      "source_concentration", "Mức tập trung vào nguồn ghi", rows=60)
fig, ax = plt.subplots(figsize=(10, max(8, len(species)*.29)))
values = concentration.largest_source_duration_pct.sort_values()
ax.barh(values.index, values, color="#a47733")
ax.axvline(80, color="#a33b32", linestyle="--", label="80%: mốc tham khảo")
ax.set(xlim=(0, 102), xlabel="% tổng độ dài clip của nhãn", title="Tỷ trọng của name lớn nhất")
ax.legend(loc="lower right")
figure(fig, "07_source_concentration", "Mốc 80% phục vụ đọc biểu đồ, không phải quy tắc tự loại lớp.")

name_audit = df[df.name.ne("")].groupby("name").agg(
    clips=("id", "size"), species_labels=("species", lambda s: s[s.ne("")].nunique()),
    devices=("device_type", lambda s: s[s.ne("")].nunique()),
    partitions=("partition", "nunique"),
    partition_names=("partition", lambda s: " | ".join(sorted(s.unique())))
).reset_index()
table(name_audit[(name_audit.species_labels > 1) | (name_audit.devices > 1) | (name_audit.partitions > 1)],
      "source_ambiguities", "Name cần truy vết thêm", rows=25)

def condition_summary(field, image_name):
    counts = pd.crosstab(mosq.species, category(mosq[field])).reindex(species.index, fill_value=0)
    table(counts.reset_index(), f"species_{field}", f"Phân bố {field} theo nhãn (số clip)", rows=8)
    keep = counts.sum().sort_values(ascending=False).head(8).index.tolist()
    if "(thiếu)" in counts and "(thiếu)" not in keep:
        keep.append("(thiếu)")
    reduced = counts[keep].copy()
    other = [c for c in counts if c not in keep]
    if other:
        reduced["(khác)"] = counts[other].sum(axis=1)
    percentages = reduced.div(reduced.sum(axis=1), axis=0)*100
    fig, ax = plt.subplots(figsize=(max(8, len(reduced.columns)*1.0), max(8, len(species)*.29)))
    sns.heatmap(percentages, vmin=0, vmax=100, cmap="Blues", ax=ax,
                cbar_kws={"label": "% clip trong từng nhãn"})
    ax.set(title=f"Nhãn × {field}", xlabel="", ylabel="")
    ax.tick_params(axis="x", rotation=35, labelsize=9)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    figure(fig, image_name, f"Chuẩn hóa theo hàng: mỗi nhãn có tổng 100%. {field} thiếu được hiển thị riêng.")
    return counts

device_counts = condition_summary("device_type", "08_species_device")
method_counts = condition_summary("method", "09_species_method")
country_counts = condition_summary("country", "10_species_country")
context_rows = []
for field in ["gender", "plurality", "fed", "age", "mic_type", "location_type"]:
    for value, group in mosq.groupby(field, dropna=False):
        context_rows.append({"field": field, "value": value or "(thiếu)",
                             "clips": len(group), "seconds": group.duration_seconds.sum()})
context_table = pd.DataFrame(context_rows)
table(context_table, "biological_recording_context", "Metadata sinh học và thu âm", rows=35)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, field in zip(axes.flat, ["gender", "plurality", "fed", "age", "mic_type", "location_type"]):
    values = context_table[context_table.field.eq(field)].nlargest(8, "clips").sort_values("clips")
    ax.barh(values.value, values.clips, color="#338777")
    ax.set(title=field, xlabel="Clip (tối đa 8 giá trị phổ biến)")
figure(fig, "11_biological_context", "Các trường thiếu vẫn nằm trong mẫu số. Không suy diễn giới tính, tuổi hoặc số cá thể từ loài.")
table(pd.crosstab(mosq.species, category(mosq.gender)).reset_index(), "species_gender", "Gender theo nhãn", rows=60)
table(pd.crosstab(mosq.species, category(mosq.plurality)).reset_index(), "species_plurality", "Plurality theo nhãn", rows=60)
few_sources = concentration[concentration.source_names.lt(3)].index.tolist()
note(f"**{len(few_sources)} nhãn có dưới 3 name khác nhau.** Danh sách: "
     + ", ".join(few_sources) + ". Ba name chỉ là điều kiện đếm sơ bộ; không đảm bảo chia được ba tập độc lập.")

## 5. Chất lượng toàn bộ WAV và tín hiệu trùng

**Làm gì:** đọc toàn bộ sample theo block để tính RMS, peak, DC offset, tỷ lệ zero và
tỷ lệ sample gần full scale; kiểm tra header và SHA-256 từng WAV. Không load cả dataset vào RAM.

- RMS dưới −60 dBFS: cờ âm lượng thấp, có thể do gain thấp, không tự kết luận không có muỗi.
- Sample có |x| ≥ 0,999: cờ gần full scale; chưa chứng minh clipping gây méo.
- SHA-256 giống nhau: file trùng byte. Chưa phát hiện mọi đoạn cắt chồng lấn, gần trùng hoặc bản resample.
- dBFS dựa trên thang PCM số, không phải mức áp suất âm thanh đã hiệu chuẩn.

Cache chỉ dùng lại khi inventory, kích thước/mtime WAV, phiên bản thuật toán và các ngưỡng
không đổi. Bật FORCE_AUDIO_SCAN để đọc/hash lại toàn bộ. Mọi cờ chỉ phục vụ kiểm tra.

In [ ]:
SECTION = 5
cache_path = OUT / "tables" / "audio_quality.csv"
cache_meta_path = OUT / "audio_quality_cache.json"
cache_settings = {"version": QC_VERSION, "near_silence_dbfs": NEAR_SILENCE_DBFS,
                  "near_full_scale": NEAR_FULL_SCALE, "soundfile": versions["soundfile"],
                  "inventory_sha256": input_hashes["inventory"]}
fingerprint = hashlib.sha256(
    (json.dumps(cache_settings, sort_keys=True) +
     states.sort_values("id").to_csv(index=False)).encode()).hexdigest()
cached_meta = json.loads(cache_meta_path.read_text()) if cache_meta_path.exists() else {}
cache_used = (not FORCE_AUDIO_SCAN and cache_path.exists() and cached_meta.get("fingerprint") == fingerprint)
if cache_used:
    qc = pd.read_csv(cache_path, dtype={"id": str}, keep_default_na=False)
    assert qc.id.is_unique and set(qc.id) == set(df.id)
    print("Dùng cache chất lượng đã đối chiếu với trạng thái file.")
else:
    quality_rows = []
    started = time.perf_counter()
    for index, row in enumerate(df.itertuples(), 1):
        result = {"id": row.id, "error": "", "sha256_matches": False, "header_matches": False}
        try:
            path = ROOT / row.relative_path
            result["sha256_matches"] = sha256(path) == row.sha256
            with sf.SoundFile(path) as audio:
                result["header_matches"] = (audio.samplerate == row.sample_rate and
                    audio.channels == row.channels and len(audio) == row.frames)
                result.update({"actual_sample_rate": audio.samplerate, "actual_channels": audio.channels,
                               "actual_frames": len(audio), "subtype": audio.subtype})
                n = zeros = near_scale = nonfinite = scalar_samples = 0
                sum_x = sum_sq = peak = 0.0
                for block in audio.blocks(blocksize=65536, dtype="float64", always_2d=True):
                    scalar_samples += block.size
                    nonfinite += int((~np.isfinite(block)).sum())
                    near_scale += int((np.abs(block) >= NEAR_FULL_SCALE).sum())
                    mono = block.mean(axis=1)
                    n += len(mono)
                    zeros += int((mono == 0).sum())
                    sum_x += float(mono.sum())
                    sum_sq += float(np.dot(mono, mono))
                    peak = max(peak, float(np.abs(mono).max(initial=0)))
                rms = math.sqrt(sum_sq/n) if n else np.nan
                result.update({"decoded_frames": n, "nonfinite_samples": nonfinite,
                    "rms": rms, "rms_dbfs": 20*np.log10(max(rms, 1e-15)),
                    "peak": peak, "dc_offset": sum_x/n if n else np.nan,
                    "zero_fraction": zeros/n if n else np.nan,
                    "near_full_scale_fraction": near_scale/scalar_samples if scalar_samples else np.nan})
                if n != row.frames or not n or nonfinite:
                    result["error"] = "Invalid decoded frame count or nonfinite samples"
        except (OSError, RuntimeError, ValueError) as exc:
            result["error"] = str(exc)
        quality_rows.append(result)
        if index % 1000 == 0:
            print(f"{index:,}/{len(df):,} WAV; {time.perf_counter()-started:.0f} giây", flush=True)
    qc = pd.DataFrame(quality_rows)
    qc.to_csv(cache_path, index=False, encoding="utf-8-sig")
    cache_meta_path.write_text(json.dumps({"fingerprint": fingerprint, "settings": cache_settings,
        "scanned_at": RUN_TIME, "elapsed_seconds": time.perf_counter()-started}, indent=2), encoding="utf-8")

for col in ["sha256_matches", "header_matches"]:
    qc[col] = qc[col].map(lambda x: str(x).lower() == "true")
table(qc, "audio_quality", "Kết quả đọc toàn bộ audio", rows=8)
bad = qc[qc.error.ne("") | ~qc.sha256_matches | ~qc.header_matches]
table(bad, "audio_errors", "Lỗi dữ liệu hoặc thay đổi nội dung", rows=20)
assert bad.empty, "Có lỗi audio/hash/header; xem bảng audio_errors trước khi tiếp tục."
q = df.merge(qc, on="id", validate="one_to_one")
q["low_level"] = q.rms_dbfs.lt(NEAR_SILENCE_DBFS)
q["all_zero"] = q.zero_fraction.eq(1)
q["near_full_scale"] = q.near_full_scale_fraction.gt(0)
quality_summary = q.groupby("sound_type").agg(
    clips=("id", "size"), all_zero=("all_zero", "sum"),
    low_level=("low_level", "sum"), near_full_scale=("near_full_scale", "sum"),
    median_rms_dbfs=("rms_dbfs", "median")).reset_index()
table(quality_summary, "quality_by_sound_type", "Cờ chất lượng theo sound_type")
flagged = q[q.low_level | q.all_zero | q.near_full_scale]
table(flagged[["id", "species", "sound_type", "relative_path", "rms_dbfs",
               "zero_fraction", "near_full_scale_fraction"]], "quality_flags", "File cần xem/nghe thêm", rows=15)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for kind, group in q.groupby("sound_type"):
    axes[0].hist(group.rms_dbfs.clip(lower=-120), bins=np.linspace(-120, 0, 61),
                 histtype="step", linewidth=1.4, label=kind)
axes[0].axvline(NEAR_SILENCE_DBFS, color="#b54b3f", linestyle="--")
axes[0].set(xlabel="RMS dBFS (giá trị < −120 gom ở −120)", ylabel="Clip", title="Mức âm lượng số")
axes[0].legend(fontsize=8)
flag_percent = pd.DataFrame({"Âm lượng thấp": q.groupby("sound_type").low_level.mean()*100,
                            "Gần full scale": q.groupby("sound_type").near_full_scale.mean()*100})
flag_percent.plot.bar(ax=axes[1], color=["#326b98", "#a47733"], rot=20)
axes[1].set(title="Tỷ lệ clip bị gắn cờ", ylabel="%", xlabel="")
figure(fig, "12_audio_quality", "Các cờ có thể chồng nhau. Chưa có file nào bị loại theo các ngưỡng này.")
fig, ax = plt.subplots(figsize=(11, max(8, len(species)*.29)))
box = ax.boxplot([q.loc[q.sound_type.eq("mosquito") & q.species.eq(label), "rms_dbfs"].to_numpy()
                  for label in order], orientation="horizontal", tick_labels=order,
                 whis=(10, 90), showfliers=False, patch_artist=True)
for patch in box["boxes"]:
    patch.set_facecolor("#c7ded7")
ax.axvline(NEAR_SILENCE_DBFS, color="#b54b3f", linestyle="--", label="Cờ RMS thấp")
ax.set(xlabel="RMS dBFS", title="Phân bố mức âm lượng số theo nhãn")
ax.legend()
figure(fig, "15_species_rms", "Mỗi quan sát là một clip. Khác mức âm lượng có thể liên quan gain/thiết bị/khoảng cách, không tự chứng minh khác biệt sinh học.")

duplicates = q[q.duplicated("sha256", keep=False)].sort_values(["sha256", "id"])
table(duplicates[["id", "species", "sound_type", "partition", "name", "relative_path", "sha256"]],
      "duplicate_audio", "Các file trùng byte", rows=30)
duplicate_groups = duplicates.groupby("sha256").agg(
    files=("id", "size"), species_labels=("species", "nunique"),
    partitions=("partition", "nunique")).reset_index()
table(duplicate_groups, "duplicate_groups", "Tóm tắt nhóm trùng")
note(f"Đọc/hash thành công **{len(qc):,} WAV**; "
     f"**{int(q.all_zero.sum())}** file toàn zero, **{int(q.low_level.sum())}** file RMS dưới "
     f"{NEAR_SILENCE_DBFS:g} dBFS, **{int(q.near_full_scale.sum())}** file có sample gần full scale. "
     f"Có **{len(duplicate_groups)} nhóm trùng byte**. Chưa kiểm tra toàn diện gần trùng/overlap.")

## 6. Nghe và nhìn tín hiệu thực tế

**Chọn mẫu:** mặc định lấy 6 nhãn bằng seed cố định rồi một clip ngẫu nhiên mỗi nhãn.
Đây là ví dụ để quan sát, không phải danh sách lớp được chọn. Sửa PREVIEW_SPECIES để xem nhãn khác.

**Hiển thị:** nghe tối đa 5 giây ở giữa clip, giữ biên độ gốc; waveform của đoạn nghe;
waveform và spectrogram tối đa 300 ms ở giữa đoạn đó. Clip ngắn được giữ đúng độ dài,
không padding. Spectrogram dùng Hann 64 ms (hoặc toàn đoạn nếu ngắn hơn), overlap 75%.
Trục tần số hiển thị đến min(4 kHz, Nyquist), không khẳng định phần ngoài 4 kHz vô ích.
Nếu đoạn ngắn chỉ tạo một khung phổ, hình sẽ có một cột trải trên thời lượng đoạn;
không diễn giải cột đó thành biến thiên theo thời gian.

Màu phổ thể hiện PSD theo dB so với 1 FS²/Hz. Thang màu cố định giúp tránh làm tín hiệu
yếu trông mạnh như tín hiệu lớn. Đường phổ nổi bật chưa chắc là tần số wingbeat cơ bản.
Phần này không ước lượng F0 và không đưa ra chẩn đoán loài bằng mắt.

In [ ]:
SECTION = 6
rng = np.random.default_rng(SEED)
available = sorted(mosq.species.unique())
preview_labels = PREVIEW_SPECIES or sorted(rng.choice(available, min(6, len(available)), replace=False).tolist())
assert set(preview_labels) <= set(available), "Có nhãn PREVIEW_SPECIES không tồn tại."
preview_rows = []
for label in preview_labels:
    candidates = mosq[mosq.species.eq(label)].sort_values("id")
    row = candidates.iloc[int(rng.integers(len(candidates)))]
    with sf.SoundFile(ROOT / row.relative_path) as audio:
        sr = audio.samplerate
        count = min(len(audio), int(5*sr))
        start = max(0, (len(audio)-count)//2)
        audio.seek(start)
        x = audio.read(count, dtype="float64", always_2d=True).mean(axis=1)
    win_length = min(len(x), int(round(WINDOW_SECONDS*sr)))
    win_start = max(0, (len(x)-win_length)//2)
    segment = x[win_start:win_start+win_length]
    preview_rows.append({"id": row.id, "species": label, "name": row["name"],
                         "sample_rate": sr, "clip_seconds": row.duration_seconds,
                         "preview_start_seconds": start/sr, "preview_seconds": len(x)/sr})
    display(Markdown(f"### {label} — ID {row.id}"))
    display(Audio(x, rate=sr, normalize=False))
    audio_buffer = io.BytesIO()
    sf.write(audio_buffer, x, sr, format="WAV", subtype="PCM_16")
    REPORT_ITEMS.append((SECTION, "audio", base64.b64encode(audio_buffer.getvalue()).decode(),
                         f"{label} — ID {row.id}; bản nghe PCM16, không normalize"))
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
    axes[0].plot(np.arange(len(x))/sr + start/sr, x, linewidth=.6, color="#326b98")
    axes[0].set(title=f"{label} | ID {row.id}", xlabel="Thời gian trong clip (s)", ylabel="Biên độ / FS")
    axes[1].plot(np.arange(len(segment))/sr*1000, segment, linewidth=.8, color="#338777")
    axes[1].set(title=f"Đoạn {len(segment)/sr*1000:.1f} ms", xlabel="ms", ylabel="Biên độ / FS")
    if len(segment) >= 8:
        nperseg = min(len(segment), max(8, int(round(.064*sr))))
        f, t, pxx = signal.spectrogram(segment, fs=sr, window="hann", nperseg=nperseg,
                                      noverlap=int(.75*nperseg), detrend=False, scaling="density")
        keep = f <= min(4000, sr/2)
        if len(t) == 1:
            time_edges = np.array([0, len(segment)/sr])
        else:
            delta_t = t[1]-t[0]
            time_edges = np.r_[t[0]-delta_t/2, (t[:-1]+t[1:])/2, t[-1]+delta_t/2]
        selected_f = f[keep]
        delta_f = f[1]-f[0]
        frequency_edges = np.r_[max(0, selected_f[0]-delta_f/2),
                                (selected_f[:-1]+selected_f[1:])/2, selected_f[-1]+delta_f/2]
        im = axes[2].pcolormesh(time_edges*1000, frequency_edges,
                               10*np.log10(np.maximum(pxx[keep], 1e-16)),
                               shading="flat", cmap="magma", vmin=-120, vmax=-30)
        axes[2].set_ylim(0, min(4000, sr/2))
        axes[2].grid(False)
        fig.colorbar(im, ax=axes[2], label="PSD dB re 1 FS²/Hz")
    axes[2].set(title="Spectrogram đoạn ngắn", xlabel="ms", ylabel="Hz")
    figure(fig, f"preview_{row.id}", f"{label}, ID {row.id}: minh họa một clip. Cần nhiều nguồn và nhiều mẫu để kết luận về nhãn.")
table(pd.DataFrame(preview_rows), "preview_samples", "Danh sách mẫu nghe và vị trí đoạn trích")

## 7. Bảng ứng viên để cùng chọn lớp

**Làm gì:** ghép lượng dữ liệu, độ phủ nguồn, khả năng dùng window, plurality và cờ
chất lượng vào một bảng. Không chấm điểm tổng hợp hoặc tự chọn top 4–6.

**Cần xem cùng nhau:**
- Có đủ dữ liệu thực và đủ nguồn có thể truy vết không?
- Một nguồn/thiết bị có chiếm phần lớn dữ liệu không?
- Nhãn có mô tả loài cụ thể hay nhóm loài?
- Metadata Single/Multiple/thiếu ảnh hưởng thế nào đến bài toán một loài?
- Bao nhiêu clip đủ 300 ms, nhưng chất lượng thực tế ra sao?

Dưới 3 name là dấu hiệu khó xây dựng ba tập theo name; từ 3 name trở lên cũng chưa
đảm bảo tính độc lập hoặc split hợp lệ. Quyết định cuối cần xem lại file nguồn.

In [ ]:
SECTION = 7
qmosq = q[q.sound_type.eq("mosquito") & q.species.ne("")].copy()
candidate = species.join(concentration.drop(columns="source_names"), how="left")
candidate["single_pct"] = qmosq.groupby("species").plurality.apply(lambda s: s.str.casefold().eq("single").mean()*100)
candidate["plurality_missing_pct"] = qmosq.groupby("species").plurality.apply(lambda s: s.eq("").mean()*100)
candidate["gender_missing_pct"] = qmosq.groupby("species").gender.apply(lambda s: s.eq("").mean()*100)
candidate["low_level_pct"] = qmosq.groupby("species").low_level.mean()*100
candidate["near_full_scale_pct"] = qmosq.groupby("species").near_full_scale.mean()*100
candidate["median_rms_dbfs"] = qmosq.groupby("species").rms_dbfs.median()
for field in ["device_type", "method", "country"]:
    candidate[f"known_{field}_values"] = qmosq.groupby("species")[field].apply(lambda s: s[s.ne("")].nunique())
    candidate[f"{field}_missing_pct"] = qmosq.groupby("species")[field].apply(lambda s: s.eq("").mean()*100)
candidate["windows_300ms_overlap50"] = window_table[window_table.window_seconds.eq(.3)].set_index("species").windows_50pct_overlap

def review_notes(row):
    reasons = []
    if row.source_names < 3:
        reasons.append("<3 name")
    if row.largest_source_duration_pct >= 80:
        reasons.append(">=80% thời lượng từ một name")
    if row.label_review:
        reasons.append("rà nhãn nhóm/complex")
    if row.plurality_missing_pct >= 50:
        reasons.append(">=50% thiếu plurality")
    if row.short_300ms_pct >= 50:
        reasons.append(">=50% clip <300 ms")
    if row.known_device_type_values <= 1:
        reasons.append("<=1 loại thiết bị đã biết")
    return "; ".join(reasons) or "Cần kiểm tra nguồn và chất lượng trước khi chọn"

candidate["review_notes"] = candidate.apply(review_notes, axis=1)
table(candidate.reset_index(), "species_candidates", "Bảng ứng viên đầy đủ — chưa chọn lớp", rows=60)
focus_cols = ["largest_source_duration_pct", "short_300ms_pct", "plurality_missing_pct",
              "gender_missing_pct", "low_level_pct", "near_full_scale_pct"]
fig, ax = plt.subplots(figsize=(11, max(8, len(candidate)*.29)))
sns.heatmap(candidate[focus_cols], vmin=0, vmax=100, annot=True, fmt=".0f", cmap="YlOrBr",
            linewidths=.3, ax=ax, cbar_kws={"label": "% (mỗi cột có định nghĩa riêng)"})
ax.set_xticklabels(["Nguồn lớn nhất\n(% thời lượng)", "Clip <300 ms", "Thiếu plurality",
                    "Thiếu gender", "RMS <−60 dBFS", "Có sample\ngần full scale"], rotation=25, ha="right")
ax.set(title="Các điểm cần cân nhắc khi chọn nhãn", xlabel="", ylabel="")
figure(fig, "13_candidate_review", "Màu đậm chỉ là tỷ lệ cao của từng tiêu chí, không phải điểm đánh giá tổng hợp. Không cộng các cột thành một thang điểm.")

## 8. Tổng hợp và phần cần thảo luận

**Report này hỗ trợ quyết định dữ liệu, chưa chứng minh khả năng phân loại.**
Các trường hợp name chung, nguồn ghi tập trung, nhãn nhóm loài và metadata thiếu cần
được xem trước khi chốt 4–6 lớp.

Sau khi thống nhất lớp, phần A tiếp tục với truy vết nhóm nguồn, split nghiên cứu,
xử lý clip ngắn/đoạn đuôi, đầu vào baseline và giao thức đánh giá. Các lựa chọn này
chưa được tự thực hiện trong notebook.

**Giới hạn:** chưa có ID cá thể/session được xác minh; chưa audit toàn diện gần trùng;
RMS và ngưỡng full scale không đo SNR hay xác nhận tín hiệu muỗi; nghe vài mẫu không
đại diện mọi bản ghi; chưa có đánh giá trên microphone/lồng thực tế.

Report HTML được tạo từ phần giải thích, bảng, hình và audio minh họa trong lần chạy này.
CSV lưu toàn bộ dòng; HTML chỉ hiện số dòng giới hạn ở các bảng dài.

In [ ]:
SECTION = 8
dev_labels = sorted(mosq.loc[mosq.partition.str.startswith("dev"), "species"].unique().tolist())
discussion = pd.DataFrame([
    {"câu hỏi": "Chọn nhãn nào?", "căn cứ": "species_candidates.csv + nghe mẫu + truy vết nguồn", "trạng thái": "Chờ thảo luận"},
    {"câu hỏi": "Nhóm độc lập dựa trên gì?", "căn cứ": "source_counts.csv và source_ambiguities.csv", "trạng thái": "Chưa xác minh cá thể/session"},
    {"câu hỏi": "Giữ clip ngắn thế nào?", "căn cứ": "window_feasibility.csv; padding không thêm tín hiệu", "trạng thái": "Chưa chốt"},
    {"câu hỏi": "Xử lý trùng và cờ chất lượng?", "căn cứ": "duplicate_audio.csv và quality_flags.csv", "trạng thái": "Chưa loại file"},
    {"câu hỏi": "Dùng dev gốc làm validation?", "căn cứ": "Độ phủ nhãn ở species_partitions.csv", "trạng thái": "Chưa tạo split nghiên cứu"}
])
table(discussion, "discussion_points", "Các quyết định cần thống nhất")
note(f"**Kết quả khảo sát:** {len(df):,} WAV; {len(mosq):,} clip mosquito có species; "
     f"{len(species)} chuỗi nhãn; {len(few_sources)} nhãn có dưới 3 name. "
     f"Dev có các nhãn mosquito: **{', '.join(dev_labels)}**. "
     "Vì độ phủ khác nhau, không tự dùng train/dev gốc làm split phân loại 4–6 lớp.")
note("**Chưa chọn loài.** Ưu tiên thảo luận bảng ứng viên và độ phủ nguồn ghi trước; "
     "không lấy số lượng window hoặc một biểu đồ đẹp làm căn cứ duy nhất.")

notebook_path = ROOT / "notebooks/01_A_data_analysis.ipynb"
notebook_data = json.loads(notebook_path.read_text(encoding="utf-8"))
code_hash = hashlib.sha256("\n".join("".join(c["source"]) for c in notebook_data["cells"]
                                    if c["cell_type"] == "code").encode()).hexdigest()
summary = {"run_time": RUN_TIME, "input_sha256": input_hashes, "notebook_code_sha256": code_hash,
           "python": platform.python_version(), "packages": versions, "settings": cache_settings,
           "seed": SEED, "window_seconds": WINDOW_SECONDS, "hop_seconds": HOP_SECONDS,
           "audio_qc_cache_used": cache_used,
           "audio_qc_scanned_at": json.loads(cache_meta_path.read_text())["scanned_at"],
           "files": len(df),
           "labeled_mosquito_clips": len(mosq), "species_label_strings": len(species),
           "missing_species_mosquito": int(mosquito_all.species.eq("").sum()),
           "labels_with_fewer_than_3_names": few_sources, "dev_species_labels": dev_labels,
           "duplicate_byte_groups": len(duplicate_groups), "audio_errors": len(bad),
           "selected_species": [], "research_split_created": False}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

parts = ['<!doctype html><html lang="vi"><meta charset="utf-8"><title>Phân tích HumBugDB — phần A</title>',
         '<style>body{font:16px/1.65 system-ui,sans-serif;max-width:1200px;margin:36px auto;padding:0 24px;color:#20313e}'
         'h1,h2{color:#245579}h2{margin-top:52px;border-bottom:1px solid #ccd6dc;padding-bottom:8px}'
         'img{max-width:100%;height:auto}table{border-collapse:collapse;font-size:13px;width:100%}'
         'th,td{border:1px solid #d8e0e5;padding:7px;text-align:left}th{background:#edf3f6}'
         '.table{overflow:auto;margin:20px 0}figure{margin:28px 0}figcaption{color:#52636d}'
         'audio{display:block;margin:12px 0 24px}a{color:#245579}</style><body>']
for section_number in range(9):
    for cell in notebook_data["cells"]:
        if cell["cell_type"] == "markdown" and cell.get("metadata", {}).get("analysis_section") == section_number:
            parts.append(mistune.html("".join(cell["source"])))
    for section_id, kind, value, caption in REPORT_ITEMS:
        if section_id != section_number:
            continue
        if kind == "text":
            parts.append(mistune.html(value))
        elif kind == "table":
            name, title, total = caption
            parts.append(f'<div class="table"><strong>{html.escape(title)}</strong>'
                         f'<p>Hiển thị {len(value)}/{total} dòng. <a href="tables/{name}.csv">CSV đầy đủ</a></p>'
                         + value.to_html(index=False, escape=True, float_format=lambda x: f"{x:,.3f}") + '</div>')
        elif kind == "figure":
            encoded = base64.b64encode(value.read_bytes()).decode()
            parts.append(f'<figure><img src="data:image/png;base64,{encoded}" alt="{html.escape(caption)}">'
                         f'<figcaption>{html.escape(caption)}</figcaption></figure>')
        elif kind == "audio":
            parts.append(f'<p>{html.escape(caption)}</p><audio controls src="data:audio/wav;base64,{value}"></audio>')
parts.append("</body></html>")
(OUT / "report.html").write_text("\n".join(parts), encoding="utf-8")
print("Report:", OUT / "report.html")
print("Bảng ứng viên:", OUT / "tables/species_candidates.csv")
print("Chưa chọn lớp hoặc tạo split nghiên cứu.")